In [3]:
# ---------------- 0. 依赖 ----------------
import numpy as np, pandas as pd
from itertools import product
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import filegetter.filegetter as fgt 

# ---------------- 1. 自激+EWMA 评估函数 ----------------
def evaluate_volume_model_selfexc(
        trades_df: pd.DataFrame,
        halflife: float,
        lookahead: int,
        alpha: float = 0.3,              # 自激强度 ∈ (0,1)
        beta: float  = 1.0,              # 衰减速度 (秒⁻¹)
        side: Optional[str] = None,      # 'buy'/'sell'/None
        return_series: bool = False):
    """
    预测未来 lookahead 秒总成交量:
        pred_t = EWMA_t + S_t
        S_t   = α*vol_{t-1} + e^{-β}*S_{t-1}

    返回 metrics 或 (metrics, pred, true) 三元组
    """
    # —— 防御式早退 ——
    if trades_df.empty:
        return (None, None, None) if return_series else None

    # —— 方向过滤 ——
    if side == "buy":
        trades_df = trades_df[trades_df["v"] > 0]
    elif side == "sell":
        trades_df = trades_df[trades_df["v"] < 0]
    if trades_df.empty:
        return (None, None, None) if return_series else None

    # —— 秒级聚合 ——
    trades_df = trades_df.copy()
    trades_df["datetime"] = pd.to_datetime(trades_df["ts"], unit="ms", utc=True)
    trades = trades_df.set_index("datetime").sort_index()
    trades["vol"] = trades["v"].abs()
    vol_1s = trades["vol"].resample("1s").sum().fillna(0)
    if len(vol_1s) <= lookahead:
        return (None, None, None) if return_series else None

    # —— baseline EWMA —— 
    ewma = vol_1s.ewm(halflife=halflife, adjust=False).mean()

    # —— 自激递推 S_t —— 
    decay = np.exp(-beta)                        # 一秒衰减
    v_arr = vol_1s.values.astype(float)
    s_arr = np.empty_like(v_arr)
    s_arr[0] = 0.0                               # S_0 = 0
    for i in range(1, len(v_arr)):
        s_arr[i] = alpha * v_arr[i-1] + decay * s_arr[i-1]
    self_exc = pd.Series(s_arr, index=vol_1s.index)

    pred = ewma + self_exc                       # 组合预测

    # —— 标签: 未来 lookahead 秒（不含当前秒） ——
    true = (vol_1s.shift(-1)
            .rolling(lookahead, min_periods=lookahead)
            .sum()
            .shift(-(lookahead - 1)))

    df_eval = pd.DataFrame({"pred": pred, "true": true}).dropna()
    if df_eval.empty:
        return (None, None, None) if return_series else None

    metrics = {
        "samples": len(df_eval),
        "MAE":   mean_absolute_error(df_eval["true"], df_eval["pred"]),
        "RMSE":  mean_squared_error(df_eval["true"], df_eval["pred"], squared=False),
        "R2":    r2_score(df_eval["true"], df_eval["pred"]),
        "zero_ratio": (df_eval["true"] == 0).mean(),   # 可选指标
    }
    if return_series:
        return metrics, df_eval["pred"], df_eval["true"]
    return metrics




In [4]:
dates = pd.date_range('2024-01-15', '2024-01-21', freq='D')
raw_list = []
for d in dates:
    ymd = d.strftime('%Y%m%d')
    try:
        raw = (fgt.get('binance', 'swap', 'btc-usdt', 'trade', ymd,
                       machine='AWS-JP1')[['p', 'v', 'ts']]
               .dropna(subset=['p','v','ts']))
        raw_list.append(raw)
        print(f'Loaded {ymd}: {len(raw):,} rows')
    except Exception as e:
        print(f'[FAIL] {ymd}: {e}')

week_df = pd.concat(raw_list, ignore_index=True)
print('Total merged rows:', len(week_df))

loading... AWS-JP1_BINANCE_SWAP_BTC-USDT_TRADE_2024_01_15.hdf 
Loaded 20240115: 1,036,180 rows
loading... AWS-JP1_BINANCE_SWAP_BTC-USDT_TRADE_2024_01_16.hdf 
Loaded 20240116: 1,181,580 rows
loading... AWS-JP1_BINANCE_SWAP_BTC-USDT_TRADE_2024_01_17.hdf 
Loaded 20240117: 898,720 rows
loading... AWS-JP1_BINANCE_SWAP_BTC-USDT_TRADE_2024_01_18.hdf 
Loaded 20240118: 1,338,520 rows
loading... AWS-JP1_BINANCE_SWAP_BTC-USDT_TRADE_2024_01_19.hdf 
Loaded 20240119: 1,332,460 rows
loading... AWS-JP1_BINANCE_SWAP_BTC-USDT_TRADE_2024_01_20.hdf 
Loaded 20240120: 421,966 rows
loading... AWS-JP1_BINANCE_SWAP_BTC-USDT_TRADE_2024_01_21.hdf 
Loaded 20240121: 314,820 rows
Total merged rows: 6524246


In [ ]:
HALFLIFES  = np.linspace(5, 60, 12)          # 5,10,…,60 秒
LOOKAHEADS = [1, 2, 3, 4, 5]                 # 秒
ALPHAS     = np.linspace(0.1, 1, 10)                      # 自激强度
BETAS      = np.linspace(0.5, 2, 4)                      # 衰减速度

best = None
for hl, la, a, b in product(HALFLIFES, LOOKAHEADS, ALPHAS, BETAS):
    m_buy  = evaluate_volume_model_selfexc(week_df, hl, la, a, b, side='buy')
    m_sell = evaluate_volume_model_selfexc(week_df, hl, la, a, b, side='sell')
    if (m_buy is None) or (m_sell is None):
        continue
    # 取买卖平均 MAE 作为目标函数
    mae_avg = 0.5 * (m_buy['MAE'] + m_sell['MAE'])
    if (best is None) or (mae_avg < best['MAE_avg']):
        best = dict(halflife=hl, lookahead=la,
                    alpha=a, beta=b, MAE_avg=mae_avg,
                    MAE_buy=m_buy['MAE'], MAE_sell=m_sell['MAE'])

print('\n>>> Week-merged最佳参数')
for k,v in best.items():
    print(f'{k:10s}: {v}')